In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
project_root = os.path.abspath("")
if project_root not in sys.path:
    sys.path.append(project_root)
import sleap
from pathlib import Path
from ipywidgets import widgets
from IPython.display import display
import matplotlib.pyplot as plt
from hypnose.utils.helpers import _get_from_cache, _update_cache
from sleap_utils import *

%matplotlib widget

INFO:numexpr.utils:Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [3]:
# Wrapper function to process all .slp files. 
# Params:
    # subjid: list of subjids to process (e.g., [40, 41, 42])
    # date: specific date or list of dates to process (e.g., [20251030, 20251231]) or None for all dates
    # node_pool: candidate SLEAP nodes to auto-select the centroid from (default None = all skeleton nodes)
    # recompute: if True, recomputes and overwrites existing output files. Keep False unless SLEAP model has been updated. 
    # base_dir: optional to set to different base dir and override symlinked default path
    # score_thresh: min confidence for a node point to count (below = treated as missing)
    # presence_frac: node must be present in >= this fraction of occupied frames to join the centroid
    # gap_limit: max consecutive missing frames to interpolate per node (longer gaps stay NaN)
sleap_processing = process_sleap_sessions(
    subjid=[66],
    date=[20260723],
    base_dir="D:",
    node_pool=None, 
    save_output=True, 
    score_thresh=0.4,
    presence_frac=0.7,
    gap_limit=120,
    recompute=True
)

Found 4 video(s) to process:
  1. 2026-07-23T11-00-14__VideoData_1904-02-13T13-00-00.predictions.slp
  2. 2026-07-23T11-00-14__VideoData_1904-02-13T14-00-00.predictions.slp
  3. 2026-07-23T11-00-14__VideoData_1904-02-13T15-00-00.predictions.slp
  4. 2026-07-23T11-00-14__VideoData_1904-02-13T16-00-00.predictions.slp

[1/4] Reading: 2026-07-23T11-00-14__VideoData_1904-02-13T13-00-00.predictions.slp
    113476 rows, frames 5023-118508

[2/4] Reading: 2026-07-23T11-00-14__VideoData_1904-02-13T14-00-00.predictions.slp
    216001 rows, frames 0-216008

[3/4] Reading: 2026-07-23T11-00-14__VideoData_1904-02-13T15-00-00.predictions.slp
    215938 rows, frames 0-216007

[4/4] Reading: 2026-07-23T11-00-14__VideoData_1904-02-13T16-00-00.predictions.slp
    174545 rows, frames 0-174558

Session node selection (score>=0.4, presence>=70% of 718,522 occupied frames):
    [x] tailbase        89.1%
    [x] center_back     88.5%
    [x] center          81.9%
    [x] neck            76.2%
    [x] left_ear

c:\Users\HarrisLab\Desktop\Repos\hypnose\sleap-hypnose\sleap_utils.py:157: RuntimeWarning: Mean of empty slice
  gx = np.nanmean(filled[xcols].to_numpy(dtype=float), axis=1)
c:\Users\HarrisLab\Desktop\Repos\hypnose\sleap-hypnose\sleap_utils.py:158: RuntimeWarning: Mean of empty slice
  gy = np.nanmean(filled[ycols].to_numpy(dtype=float), axis=1)


  ✓ Saved sleap_tracking_video1_2026-07-23T11-00-14__VideoData_1904-02-13T13-00-00.parquet: 113476 rows, 113474 frames with centroid
  ✓ Saved sleap_tracking_video2_2026-07-23T11-00-14__VideoData_1904-02-13T14-00-00.parquet: 216001 rows, 216001 frames with centroid
  ✓ Saved sleap_tracking_video3_2026-07-23T11-00-14__VideoData_1904-02-13T15-00-00.parquet: 215938 rows, 215938 frames with centroid
  ✓ Saved sleap_tracking_video4_2026-07-23T11-00-14__VideoData_1904-02-13T16-00-00.parquet: 174545 rows, 174545 frames with centroid

✅ All videos processed!

Subject 66 Date 20260723 - Processing SLEAP Output:
  Skipped: No subject directory found for sub-066

Summary:
  66:
    Successful: 0/1
    Failed: 0/1
    Skipped: 1/1


In [4]:
quality_report = sleap_node_quality_report(
    subjid=66,
    date=20260723,
)

20260723: no .slp files, skipping


## Step by Step .slp file processing (use above wrapper for multiple subjids/dates)

In [ ]:
# 1. Step: extract frames from .slp files and save with centroid coordinates in csv files per video

subjid = 45
date = 20251209

sleap_data = sleap_labels_and_centroid(subjid=subjid, date=date, skip_empty=True)

In [ ]:
# 2. Step: add timestamps to all video tracking csv files and combine into single dataframe
tracking_times = add_timestamps_to_sleap_tracking(subjid=45, date=20251209)

## Video Creation: Create an annotated video with centroid overlay and odor presentation + reward status information

In [6]:
output_video = annotate_videos_with_sleap_and_trials(
    subjid=66,
    date=20260723,
    rotate_deg=90,
    base_dir="D:/",
    time_window=("00:05:00", "00:07:00"), 
    video_indices=None, # which videos to annotate, can be multiple by passing [1, 2, 3] or all by passing None
    reward_display_s=2,
    mark_timepoint="11:33:11.000"
)
# good example video quintuples: 40, 20251120, index 2, 0:10:38 to 0:11:10 (3 trials, B, A, B)

FileNotFoundError: No combined timestamps file found in D:\derivatives\sub-066_id-427\ses-004_date-20260723\saved_analysis_results

In [ ]:
sleap_quality = sleap_node_quality_report(
    subjid=57,
    date=20260622,
    base_dir="E:/",
    score_thresh=0.4,
    presence_frac=0.7,
    verbose=True
)

# Miscellaneous 

In [ ]:
# Cell to check what real time (e.g., sequence_start) relates to what video index and time from start of video, to help set parameters for annotate_videos_with_sleap_and_trials time_window and video_indices
from pathlib import Path
import pandas as pd
from math import floor, ceil
from hypnose.io.paths import get_derivatives_root
from hypnose.utils.helpers import _get_from_cache, _update_cache

# Given absolute clock times, find which video covers that window and compute per-video offsets
subjid = 40
date = 20251128
start_time_str = "16:29:45"  # HH:MM:SS (rounded down to this second)
end_time_str   = "16:30:15"  # HH:MM:SS (rounded up to this second)

# Helper for formatting seconds (available both paths)
def _fmt_hhmmss(seconds):
    seconds = max(seconds, 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# Try cached summary to avoid re-reading CSVs
cache_kind = "sleap_timestamps"
cached = _get_from_cache(subjid, date, kind=cache_kind)
if cached is not None:
    df = cached.get("df") if isinstance(cached, dict) else cached
    combined_path = cached.get("combined_path") if isinstance(cached, dict) else None
else:
    cached = None

if cached is None or df is None:
    date_str = str(date)
    sub_str = f"sub-{subjid:03d}"
    deriv_root = get_derivatives_root()
    sub_dirs = list(deriv_root.glob(f"{sub_str}_id-*"))
    if not sub_dirs:
        raise FileNotFoundError(f"No subject dir for {sub_str} under {deriv_root}")
    ses_dirs = list(sub_dirs[0].glob(f"ses-*_date-{date_str}"))
    if not ses_dirs:
        raise FileNotFoundError(f"No session dir for date {date_str}")
    results_dir = ses_dirs[0] / "saved_analysis_results"

    combined_files = list(results_dir.glob("*_combined_sleap_tracking_timestamps.parquet")) or list(results_dir.glob("*_combined_sleap_tracking_timestamps.csv"))
    if not combined_files:
        raise FileNotFoundError("No combined timestamps file found; run add_timestamps_to_sleap_tracking first")
    combined_path = combined_files[0]

    df = pd.read_parquet(combined_path) if combined_path.suffix == ".parquet" else pd.read_csv(combined_path)
    if "time" not in df.columns or "video_file" not in df.columns:
        raise ValueError("Combined CSV missing required columns 'time' or 'video_file'")

    # Normalize times to naive for comparison
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["time_naive"] = df["time"].dt.tz_localize(None)

    # Cache dataframe and path for reuse
    _update_cache(subjid, [date], {date: {"df": df, "combined_path": combined_path}}, kind=cache_kind)
else:
    # Cached df may be missing time_naive; ensure present
    if "time_naive" not in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df["time_naive"] = df["time"].dt.tz_localize(None)

date_str = str(date)
sub_str = f"sub-{subjid:03d}"
if 'deriv_root' not in locals():
    deriv_root = get_derivatives_root()
sub_dirs = list(deriv_root.glob(f"{sub_str}_id-*"))
if not sub_dirs:
    raise FileNotFoundError(f"No subject dir for {sub_str} under {deriv_root}")
ses_dirs = list(sub_dirs[0].glob(f"ses-*_date-{date_str}"))
if not ses_dirs:
    raise FileNotFoundError(f"No session dir for date {date_str}")
results_dir = ses_dirs[0] / "saved_analysis_results"

# Order videos by first appearance in the combined file
video_order = []
seen = set()
for vf in df["video_file"]:
    if vf not in seen:
        video_order.append(vf)
        seen.add(vf)
video_index_map = {vf: idx + 1 for idx, vf in enumerate(video_order)}  # 1-based

summary = []
for vf, sub in df.groupby("video_file"):
    t_min = sub["time_naive"].min()
    t_max = sub["time_naive"].max()
    summary.append((vf, t_min, t_max))

start_dt = pd.to_datetime(f"{date_str} {start_time_str}", errors="coerce")
end_dt = pd.to_datetime(f"{date_str} {end_time_str}", errors="coerce")
if pd.isna(start_dt) or pd.isna(end_dt):
    raise ValueError("Invalid start/end times: could not parse")
if end_dt <= start_dt:
    # Auto-swap if user accidentally reversed times
    print("Note: end_time precedes start_time; swapping them for the lookup")
    start_dt, end_dt = end_dt, start_dt

# Global recording bounds for sanity check
rec_min = df["time_naive"].min()
rec_max = df["time_naive"].max()
if start_dt < rec_min or end_dt > rec_max:
    print(f"Warning: requested window [{start_dt} .. {end_dt}] extends outside recorded range [{rec_min} .. {rec_max}]")

# Find the video that contains the start time
matches = [item for item in summary if item[1] <= start_dt <= item[2]]
if not matches:
    print("No video covers the requested start time.")
else:
    vf, t_min, t_max = matches[0]
    idx = video_index_map.get(vf, None)
    # Round start down to whole second, end up to whole second (but not past video end)
    start_offset_raw = (start_dt - t_min).total_seconds()
    end_offset_raw = (min(end_dt, t_max) - t_min).total_seconds()
    start_offset = floor(start_offset_raw)
    end_offset = ceil(end_offset_raw)
    print(f"Video file: {vf}")
    if idx is not None:
        print(f"Video index (1-based): {idx}")
    print(f"Time window for annotate_videos_with_sleap_and_trials: (\"{_fmt_hhmmss(start_offset)}\", \"{_fmt_hhmmss(end_offset)}\")")
    print(f"Video covers {t_min} to {t_max} (clock time)")
    if end_dt > t_max:
        over = (end_dt - t_max).total_seconds()
        print(f"Warning: requested end extends {over:.2f}s past this video; window clipped to video end")


In [ ]:
# Inspect Port0 events in a time window (raw digital input) --> get all Poke IN and OUT events in the time window
import pandas as pd
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 40
date = 20251128  # YYYYMMDD
start_time_str = "15:45:22"  # HH:MM:SS or HH:MM:SS.mmm
end_time_str   = "15:45:23"  # HH:MM:SS or HH:MM:SS.mmm
experiment_index = 0  # if multiple runs exist for the same date
# ----------------------

# Resolve experiment root using the same helper as classification_utils
exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; check subjid/date/index")
print(f"Using experiment root: {exp_root}")

# Load streams (uses heartbeat correction internally); verbose=True to show progress
streams = load_all_streams(exp_root, apply_corrections=True, verbose=True)
port0 = streams.get("digital_input_data", {}).get("DIPort0")
if port0 is None or port0.empty:
    raise ValueError("DIPort0 not found or empty")
port0 = port0.astype(bool).sort_index()

# Define window
start_dt = pd.to_datetime(f"{date} {start_time_str}", errors="coerce")
end_dt = pd.to_datetime(f"{date} {end_time_str}", errors="coerce")
if pd.isna(start_dt) or pd.isna(end_dt):
    raise ValueError("Could not parse start/end times")
if end_dt <= start_dt:
    raise ValueError("end_time must be after start_time")

# Slice and find edges
slice_ser = port0.loc[start_dt:end_dt]
if slice_ser.empty:
    raise ValueError("No Port0 samples in the requested window; adjust times")

rises = slice_ser & ~slice_ser.shift(1, fill_value=False)
falls = ~slice_ser & slice_ser.shift(1, fill_value=False)

rise_times = rises[rises].index.to_list()
fall_times = falls[falls].index.to_list()

print(f"Port0 window: {start_dt} -> {end_dt}")
print(f"N samples in window: {len(slice_ser)}")
print(f"Rising edges (poke-in): {len(rise_times)}")
for t in rise_times:
    print(f"  IN  @ {t}")
print(f"Falling edges (poke-out): {len(fall_times)}")
for t in fall_times:
    print(f"  OUT @ {t}")


In [ ]:
# Get the nearest Poke OUT for a given timestamp, and the video frame (rawdata) that is closes to that timestamp. 
import pandas as pd
from pathlib import Path
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 99
date = 20260312
experiment_index = 3  # pick run if multiple
port0_event_ts = "2026-03-12T10:11:59.574432"  # rising edge (clock-corrected)
target_video_name = None # filter to this AVI; set None to search all
# ----------------------

# Resolve experiment root and load raw streams (device-time + folder offset)
exp_root = load_experiment(subjid, date, index=experiment_index)
streams = load_all_streams(exp_root, apply_corrections=True, verbose=False)

# Port0 falling edges
port0 = streams.get("digital_input_data", {}).get("DIPort0")
if port0 is None or port0.empty:
    raise ValueError("DIPort0 missing")
port0 = port0.astype(bool).sort_index()
falls = (~port0) & port0.shift(1, fill_value=False)
fall_times = falls[falls].index

# Find nearest falling edge to the provided timestamp
event_ts = pd.to_datetime(port0_event_ts, errors="coerce")
if pd.isna(event_ts):
    raise ValueError("Could not parse port0_event_ts")
# TimedeltaIndex lacks .abs; use numpy absolute
nearest_edge = (pd.Index(abs(fall_times - event_ts))).argmin()
edge_ts = fall_times[nearest_edge]
edge_delta_s = (edge_ts - event_ts).total_seconds()
print(f"Requested Port0 OUT: {event_ts}")
print(f"Nearest Port0 OUT:   {edge_ts} (delta {edge_delta_s:+.6f} s)")

# Video metadata
video_df = streams.get("video_data")
if video_df is None or video_df.empty:
    raise ValueError("video_data missing")
video_df = video_df.copy()

# Optional filter to a specific AVI
if target_video_name:
    video_df = video_df[video_df['_path'].astype(str).str.endswith(target_video_name)]
    if video_df.empty:
        raise ValueError(f"No video_data rows for {target_video_name}")

# Ensure hardware columns present
if 'hw_counter' not in video_df.columns:
    raise ValueError("hw_counter column missing in video_data")

# Find nearest frame by timestamp
nearest_frame_idx = (pd.Index(abs(video_df.index - edge_ts))).argmin()
frame_row = video_df.iloc[nearest_frame_idx]
raw_hw_counter = int(frame_row['hw_counter']) if pd.notna(frame_row['hw_counter']) else None
first_hw_counter = int(video_df['hw_counter'].iloc[0]) if pd.notna(video_df['hw_counter'].iloc[0]) else None
corrected_frame_id = raw_hw_counter - first_hw_counter if raw_hw_counter is not None and first_hw_counter is not None else None
time_delta_s = (frame_row.name - edge_ts).total_seconds()

print("\nNearest frame to Port0 OUT:")
print(f"  video path: {frame_row.get('_path')}")
print(f"  frame time: {frame_row.name}")
print(f"  hw_counter (raw): {raw_hw_counter}")
print(f"  hw_counter (corrected to start): {corrected_frame_id}")
print(f"  time delta vs Port0 OUT: {time_delta_s:+.6f} s")
if raw_hw_counter is not None:
    count_inclusive = (video_df['hw_counter'] <= raw_hw_counter).sum()
    print(f"  frames up to and including this hw_counter: {count_inclusive}")



In [ ]:
# Hybrid alignment: match input timestamp in corrected time, then map to raw time for frame/index matching
import pandas as pd
import numpy as np
import harp
import hypnose.trial_classification.classification_utils as cu
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 45
date = 20260205
experiment_index = 1
port0_event_ts = "2026-02-05T15:47:20.699072"  # this is in corrected time
target_video_name = "VideoData_1904-01-01T03-00-00.avi"  # set None to use all videos
# ----------------------

exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; check subjid/date/index")

# 1) Load corrected streams for matching the user-provided timestamp
streams_corr = load_all_streams(exp_root, apply_corrections=True, verbose=False)
port0_corr = streams_corr.get("digital_input_data", {}).get("DIPort0")
if port0_corr is None or port0_corr.empty:
    raise ValueError("Corrected DIPort0 missing")
port0_corr = port0_corr.astype(bool).sort_index()
falls_corr = (~port0_corr) & port0_corr.shift(1, fill_value=False)
fall_times_corr = falls_corr[falls_corr].index
if len(fall_times_corr) == 0:
    raise ValueError("No corrected Port0 falling edges found")

event_ts = pd.to_datetime(port0_event_ts, errors="coerce")
if pd.isna(event_ts):
    raise ValueError("Could not parse port0_event_ts")
nearest_corr_idx = (pd.Index(abs(fall_times_corr - event_ts))).argmin()
edge_ts_corr = fall_times_corr[nearest_corr_idx]
delta_corr_s = (edge_ts_corr - event_ts).total_seconds()

print(f"Requested corrected timestamp: {event_ts}")
print(f"Nearest CORRECTED Port0 OUT:  {edge_ts_corr} (delta {delta_corr_s:+.6f} s)")
print(f"Corrected Port0 OUT edge index: {nearest_corr_idx}")

# 2) Load raw streams for frame/index work
streams_raw = load_all_streams(exp_root, apply_corrections=False, verbose=False)
port0_raw = streams_raw.get("digital_input_data", {}).get("DIPort0")
if port0_raw is None or port0_raw.empty:
    raise ValueError("Raw DIPort0 missing")
port0_raw = port0_raw.astype(bool).sort_index()
falls_raw = (~port0_raw) & port0_raw.shift(1, fill_value=False)
fall_times_raw = falls_raw[falls_raw].index
if len(fall_times_raw) == 0:
    raise ValueError("No raw Port0 falling edges found")

# Map corrected nearest edge -> raw edge by edge index
if nearest_corr_idx >= len(fall_times_raw):
    raise IndexError(
        f"Edge-index mapping failed: corrected edge index {nearest_corr_idx} exceeds raw edge count {len(fall_times_raw)}"
    )
edge_ts_raw = fall_times_raw[nearest_corr_idx]
print(f"Mapped RAW Port0 OUT (same edge index): {edge_ts_raw}")

# Raw video metadata
video_df = streams_raw.get("video_data")
if video_df is None or video_df.empty:
    raise ValueError("Raw video_data missing")
video_df = video_df.copy()
if target_video_name:
    video_df = video_df[video_df["_path"].astype(str).str.endswith(target_video_name)]
    if video_df.empty:
        raise ValueError(f"No raw video_data rows for {target_video_name}")
if "hw_counter" not in video_df.columns:
    raise ValueError("hw_counter column missing in raw video_data")

# Nearest raw frame to mapped raw Port0 OUT
nearest_frame_raw_idx = (pd.Index(abs(video_df.index - edge_ts_raw))).argmin()
frame_row = video_df.iloc[nearest_frame_raw_idx]
raw_hw_counter = int(frame_row["hw_counter"]) if pd.notna(frame_row["hw_counter"]) else None
first_hw_counter = int(video_df["hw_counter"].iloc[0]) if pd.notna(video_df["hw_counter"].iloc[0]) else None
corrected_hw_counter = (raw_hw_counter - first_hw_counter) if raw_hw_counter is not None and first_hw_counter is not None else None
hw_counter_index = (video_df["hw_counter"] <= raw_hw_counter).sum() if raw_hw_counter is not None else None
delta_frame_raw_s = (frame_row.name - edge_ts_raw).total_seconds()

print("\nNearest RAW video frame to mapped RAW Port0 OUT:")
print(f"  video path: {frame_row.get('_path')}")
print(f"  frame time: {frame_row.name}")
print(f"  hw_counter (raw): {raw_hw_counter}")
print(f"  hw_counter (corrected to start): {corrected_hw_counter}")
print(f"  hw_counter index / frames up to this row: {hw_counter_index}")
print(f"  time delta vs RAW Port0 OUT: {delta_frame_raw_s:+.6f} s")

# Raw stream 92 = Camera0Frame (Behavior register address 92)
behavior_reader = harp.create_reader(str(cu.BEHAVIOR_SCHEMA_PATH), epoch=harp.REFERENCE_EPOCH)
stream92 = cu.load(behavior_reader.Camera0Frame, exp_root / "Behavior")
if stream92 is None or stream92.empty:
    raise ValueError("Raw stream 92 (Camera0Frame) missing or empty")
stream92 = stream92.sort_index()

# Nearest raw stream 92 event to mapped raw Port0 OUT
nearest_92_idx = (pd.Index(abs(stream92.index - edge_ts_raw))).argmin()
stream92_ts = stream92.index[nearest_92_idx]
stream92_delta_s = (stream92_ts - edge_ts_raw).total_seconds()

print("\nNearest RAW stream 92 event to mapped RAW Port0 OUT:")
print(f"  stream92 time: {stream92_ts}")
print(f"  stream92 positional index: {nearest_92_idx}")
print(f"  time delta vs RAW Port0 OUT: {stream92_delta_s:+.6f} s")

# Nearest raw video frame to that stream 92 event
nearest_frame_to_92_idx = (pd.Index(abs(video_df.index - stream92_ts))).argmin()
frame92_row = video_df.iloc[nearest_frame_to_92_idx]
raw_hw_counter_92 = int(frame92_row["hw_counter"]) if pd.notna(frame92_row["hw_counter"]) else None
corrected_hw_counter_92 = (raw_hw_counter_92 - first_hw_counter) if raw_hw_counter_92 is not None and first_hw_counter is not None else None
hw_counter_index_92 = (video_df["hw_counter"] <= raw_hw_counter_92).sum() if raw_hw_counter_92 is not None else None
delta_92_frame_s = (frame92_row.name - stream92_ts).total_seconds()

print("\nRAW video frame matching nearest RAW stream 92 event:")
print(f"  frame time: {frame92_row.name}")
print(f"  hw_counter (raw): {raw_hw_counter_92}")
print(f"  hw_counter (corrected to start): {corrected_hw_counter_92}")
print(f"  hw_counter index / frames up to this row: {hw_counter_index_92}")
print(f"  time delta vs RAW stream92 event: {delta_92_frame_s:+.6f} s")
print(video_df[["hw_counter"]].head(5))
print(f"Raw Stream 92 Head:{stream92.head(5)}")

In [ ]:
# inspect rise times: 
# Get the nearest Poke IN for a given timestamp, and the video frame (rawdata) that is closes to that timestamp. 
import pandas as pd
from pathlib import Path
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 99
date = 20260312
experiment_index = 0  # pick run if multiple
port0_event_ts = "2026-03-12T10:05:27.838400"  # rising edge (clock-corrected)
target_video_name = None # filter to this AVI; set None to search all
# ----------------------

# Resolve experiment root and load raw streams (device-time + folder offset)
exp_root = load_experiment(subjid, date, index=experiment_index)
streams = load_all_streams(exp_root, apply_corrections=True, verbose=False)

# Port0 falling edges
port0 = streams.get("digital_input_data", {}).get("DIPort0")
if port0 is None or port0.empty:
    raise ValueError("DIPort0 missing")
port0 = port0.astype(bool).sort_index()
rises = port0 & ~port0.shift(1, fill_value=False)
rise_times = rises[rises].index


# Find nearest rising edge to the provided timestamp
event_ts = pd.to_datetime(port0_event_ts, errors="coerce")
if pd.isna(event_ts):
    raise ValueError("Could not parse port0_event_ts")
# TimedeltaIndex lacks .abs; use numpy absolute
nearest_edge = (pd.Index(abs(rise_times - event_ts))).argmin()
edge_ts = rise_times[nearest_edge]
edge_delta_s = (edge_ts - event_ts).total_seconds()
print(f"Requested Port0 OUT: {event_ts}")
print(f"Nearest Port0 OUT:   {edge_ts} (delta {edge_delta_s:+.6f} s)")

# Video metadata
video_df = streams.get("video_data")
if video_df is None or video_df.empty:
    raise ValueError("video_data missing")
video_df = video_df.copy()

# Optional filter to a specific AVI
if target_video_name:
    video_df = video_df[video_df['_path'].astype(str).str.endswith(target_video_name)]
    if video_df.empty:
        raise ValueError(f"No video_data rows for {target_video_name}")

# Ensure hardware columns present
if 'hw_counter' not in video_df.columns:
    raise ValueError("hw_counter column missing in video_data")

# Find nearest frame by timestamp
nearest_frame_idx = (pd.Index(abs(video_df.index - edge_ts))).argmin()
frame_row = video_df.iloc[nearest_frame_idx]
raw_hw_counter = int(frame_row['hw_counter']) if pd.notna(frame_row['hw_counter']) else None
first_hw_counter = int(video_df['hw_counter'].iloc[0]) if pd.notna(video_df['hw_counter'].iloc[0]) else None
corrected_frame_id = raw_hw_counter - first_hw_counter if raw_hw_counter is not None and first_hw_counter is not None else None
time_delta_s = (frame_row.name - edge_ts).total_seconds()

print("\nNearest frame to Port0 OUT:")
print(f"  video path: {frame_row.get('_path')}")
print(f"  frame time: {frame_row.name}")
print(f"  hw_counter (raw): {raw_hw_counter}")
print(f"  hw_counter (corrected to start): {corrected_frame_id}")
print(f"  time delta vs Port0 OUT: {time_delta_s:+.6f} s")
if raw_hw_counter is not None:
    count_inclusive = (video_df['hw_counter'] <= raw_hw_counter).sum()
    print(f"  frames up to and including this hw_counter: {count_inclusive}")



In [ ]:
# Inspect OutputClear (register 35) events, heartbeat, and video metadata summaries
import pandas as pd
from pathlib import Path
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

subjid = 45
date = 20260205
experiment_index = 1
apply_corrections = True
video_csv_index = 0  # choose which VideoData CSV to summarize

# Resolve experiment root and pull all streams
exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; check subjid/date/index")
streams = load_all_streams(exp_root, apply_corrections=apply_corrections, verbose=False)

output_clear = streams.get("output_clear")
if output_clear is None or output_clear.empty:
    raise ValueError("output_clear stream missing or empty; verify Behavior register 35 data exists")

print(f"output_clear shape: {output_clear.shape}")
print("Columns:", list(output_clear.columns))
print(output_clear.head())

heartbeat = streams.get("heartbeat")
if heartbeat is None or heartbeat.empty:
    raise ValueError("heartbeat stream missing or empty; verify TimestampSeconds data exists")
n_rows, n_cols = heartbeat.shape
print(f"\nheartbeat shape: {n_rows} rows x {n_cols} columns")
print(heartbeat.head(20))

video_dir = exp_root / "VideoData"
video_csvs = sorted(video_dir.glob("*.csv"))
if not video_csvs:
    raise FileNotFoundError(f"No VideoData CSV files found in {video_dir}")
if not (0 <= video_csv_index < len(video_csvs)):
    raise IndexError(f"video_csv_index {video_csv_index} out of range; found {len(video_csvs)} files")
video_csv_path = video_csvs[video_csv_index]
video_csv = pd.read_csv(video_csv_path)
if "Seconds" not in video_csv.columns:
    raise ValueError(f"Column 'Seconds' missing in {video_csv_path.name}; cannot summarize start/end seconds")

video_rows = len(video_csv)
start_seconds = video_csv["Seconds"].iloc[0] if video_rows else None
end_seconds = video_csv["Seconds"].iloc[-1] if video_rows else None
corrected_time = video_rows / 60.0

print(f"\nVideo CSV ({video_csv_path.name}):")
print(f"  rows: {video_rows}")
print(f"  start Seconds: {start_seconds}")
print(f"  end Seconds: {end_seconds}")
print(f"  corrected_time (rows / 60): {corrected_time:.3f} s")

heartbeat_start_second = heartbeat["TimestampSeconds"].iloc[0] if n_rows else None
heartbeat_end_second = heartbeat["TimestampSeconds"].iloc[-1] if n_rows else None
print(f"\nHeartbeat stream covers seconds {heartbeat_start_second} to {heartbeat_end_second}")

output_set = streams.get("output_set")
if output_set is None or output_set.empty:
    raise ValueError("output_set stream missing or empty; verify Behavior register 34 data exists")
print(f"\noutput_set shape: {output_set.shape}")
print("Columns:", list(output_set.columns))
print(output_set.head())

In [ ]:
# Summarize key register states and corrected timestamps
import pandas as pd
import numpy as np

required_vars = {
    "streams": "streams",
    "output_set": "output_set",
    "output_clear": "output_clear",
    "video_csv": "video_csv",
    "heartbeat": "heartbeat",
}
missing = [name for name, var in required_vars.items() if var not in globals()]
if missing:
    raise NameError(f"Missing variables from previous cell: {missing}. Please run the prior cell first.")

def _ensure_series(name, df):
    if df is None or df.empty:
        raise ValueError(f"{name} is empty; rerun the previous cell to load data")
    return df

output_set_df = _ensure_series("output_set", output_set)
output_clear_df = _ensure_series("output_clear", output_clear)
video_csv_df = _ensure_series("video_csv", video_csv)
heartbeat_df = _ensure_series("heartbeat", heartbeat)

def _get_column(df, column_name):
    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' missing from dataframe")
    return df[column_name]

def _interp_timestamp(seconds_value):
    ts_series = streams.get("timestamp_to_time")
    if not isinstance(ts_series, pd.Series) or ts_series.empty or pd.isna(seconds_value):
        return pd.NaT
    ts_series = ts_series.dropna().sort_index()
    if ts_series.empty:
        return pd.NaT
    sec_idx = ts_series.index.to_numpy(dtype=float)
    time_ns = ts_series.values.astype("datetime64[ns]").astype(np.int64)
    if seconds_value <= sec_idx[0]:
        delta_ns = int((seconds_value - sec_idx[0]) * 1e9)
        return pd.to_datetime(time_ns[0] + delta_ns)
    if seconds_value >= sec_idx[-1]:
        delta_ns = int((seconds_value - sec_idx[-1]) * 1e9)
        return pd.to_datetime(time_ns[-1] + delta_ns)
    interp_ns = np.interp(seconds_value, sec_idx, time_ns)
    return pd.to_datetime(int(interp_ns))

def _correct_seconds(seconds_value):
    corrected = _interp_timestamp(seconds_value)
    real_offset = streams.get("real_time_offset", pd.Timedelta(0))
    if isinstance(corrected, pd.Timestamp) and pd.notna(corrected):
        corrected = corrected + real_offset
    return corrected

def _correct_timestamp(ts_value):
    if pd.isna(ts_value):
        return pd.NaT
    ts = pd.to_datetime(ts_value, errors="coerce")
    if pd.isna(ts):
        return pd.NaT
    return ts + streams.get("real_time_offset", pd.Timedelta(0))

output_set_first_time = pd.to_datetime(output_set_df.index[0], errors="coerce")
output_clear_last_time = pd.to_datetime(output_clear_df.index[-1], errors="coerce")

video_seconds_first = float(video_csv_df["Seconds"].iloc[0])
video_seconds_second = float(video_csv_df["Seconds"].iloc[1]) if len(video_csv_df) > 1 else None
video_seconds_last = float(video_csv_df["Seconds"].iloc[-1])
video_time_first = _correct_seconds(video_seconds_first)
video_time_second = _correct_seconds(video_seconds_second) if video_seconds_second is not None else None
video_time_last = _correct_seconds(video_seconds_last)

if "Time" in heartbeat_df.columns:
    hb_times = heartbeat_df["Time"]
elif heartbeat_df.index.name == "Time" or pd.api.types.is_datetime64_any_dtype(heartbeat_df.index):
    hb_times = heartbeat_df.index
else:
    hb_times = pd.Series([pd.NaT] * len(heartbeat_df))

heartbeat_time_first = _correct_timestamp(hb_times.iloc[0])
heartbeat_time_last = _correct_timestamp(hb_times.iloc[-1])

print("Register summary:")
print(f"  output_set first DOPort0 timestamp: {output_set_first_time}")
print(f"  output_clear last DOPort0 timestamp: {output_clear_last_time}")

print("\nVideo CSV time-corrected bounds:")
print(f"  first row seconds: {video_seconds_first} -> corrected {video_time_first}")
print(f"  second row seconds: {video_seconds_second} -> corrected {video_time_second}")
print(f"  last row seconds:  {video_seconds_last} -> corrected {video_time_last}")

print("\nHeartbeat time-corrected bounds:")
print(f"  first Time entry corrected: {heartbeat_time_first}")
print(f"  last Time entry corrected:  {heartbeat_time_last}")

In [ ]:
# Compare corrected vs raw heartbeat/video/output_set/output_clear timestamps
import pandas as pd
import numpy as np
import harp
import hypnose.trial_classification.classification_utils as cu
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# --- Configure session ---
subjid = 99
date = 20260312
experiment_index = 3
video_csv_index = 0
n_points = 3  # number of entries to report for heartbeat/video
# -------------------------

exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; update the configuration above.")

streams = load_all_streams(exp_root, apply_corrections=True, verbose=False)

timestamp_map = streams.get("timestamp_to_time")
real_time_offset = streams.get("real_time_offset", pd.Timedelta(0))

def _correct_seconds(seconds_value):
    """Map hardware seconds to corrected wall time using heartbeat interpolation."""
    if not isinstance(timestamp_map, pd.Series) or timestamp_map.empty or pd.isna(seconds_value):
        return pd.NaT
    ts_series = timestamp_map.dropna().sort_index()
    if ts_series.empty:
        return pd.NaT
    sec_idx = ts_series.index.to_numpy(dtype=float)
    time_ns = ts_series.values.astype("datetime64[ns]").astype(np.int64)
    sec_val = float(seconds_value)
    if len(sec_idx) == 1:
        base_ns = time_ns[0] + int((sec_val - sec_idx[0]) * 1e9)
    elif sec_val <= sec_idx[0]:
        base_ns = time_ns[0] + int((sec_val - sec_idx[0]) * 1e9)
    elif sec_val >= sec_idx[-1]:
        base_ns = time_ns[-1] + int((sec_val - sec_idx[-1]) * 1e9)
    else:
        base_ns = np.interp(sec_val, sec_idx, time_ns)
    return pd.to_datetime(int(base_ns)) + real_time_offset

def _split_first_last(df, count):
    if df is None or len(df) == 0:
        return pd.DataFrame(), pd.DataFrame()
    count = min(count, len(df))
    return df.head(count).copy(), df.tail(count).copy()

def _available_columns(df, requested):
    return [col for col in requested if col in df.columns]

def _print_table(title, frame, columns):
    print(f"\n{title}")
    if frame.empty or not columns:
        print("  (no data)")
    else:
        print(frame.loc[:, columns].to_string(index=False))

def _register_edge(df, which="first", fallback=None):
    if df is None or df.empty or "DOPort0" not in df.columns:
        return None
    pos = 0 if which == "first" else -1
    timestamp = df.index[pos] if len(df.index) else None
    row = df.iloc[pos]
    seconds_val = row.get("Seconds")
    if (seconds_val is None or pd.isna(seconds_val)) and fallback is not None and not fallback.empty and "Seconds" in fallback.columns:
        seconds_val = fallback.iloc[pos].get("Seconds")
    return {
        "timestamp": timestamp,
        "seconds": seconds_val,
        "state": bool(row.get("DOPort0", False))
    }

def _print_edge(label, edge):
    if edge is None:
        print(f"{label}: (no data)")
        return
    print(f"{label}: time={edge['timestamp']}, seconds={edge['seconds']}, DOPort0={edge['state']}")

# Locate raw files
video_dir = exp_root / "VideoData"
video_csvs = sorted(video_dir.glob("*.csv"))
if not video_csvs:
    raise FileNotFoundError(f"No VideoData CSV files found in {video_dir}")
if not (0 <= video_csv_index < len(video_csvs)):
    raise IndexError(f"video_csv_index {video_csv_index} out of range (found {len(video_csvs)} files)")
video_csv_path = video_csvs[video_csv_index]
video_csv = pd.read_csv(video_csv_path)
if "Seconds" not in video_csv.columns:
    raise ValueError(f"'Seconds' column missing in {video_csv_path.name}")
video_csv["Seconds"] = pd.to_numeric(video_csv["Seconds"], errors="coerce")
video_timestamp_col = next((col for col in video_csv.columns if "Timestamp" in col), None)

# Load raw heartbeat/output data without corrections
behavior_reader = harp.create_reader(str(cu.BEHAVIOR_SCHEMA_PATH), epoch=harp.REFERENCE_EPOCH)
raw_heartbeat = cu.load(behavior_reader.TimestampSeconds, exp_root / "Behavior")
raw_output_set = cu.load(behavior_reader.OutputSet, exp_root / "Behavior")
raw_output_clear = cu.load(behavior_reader.OutputClear, exp_root / "Behavior")
if not raw_heartbeat.empty:
    raw_heartbeat = raw_heartbeat.reset_index().rename(columns={"index": "Time"})
for df in (raw_output_set, raw_output_clear):
    if df is not None and not df.empty and "Seconds" in df.columns:
        df["Seconds"] = pd.to_numeric(df["Seconds"], errors="coerce")

# Prepare corrected heartbeat summary
heartbeat_df = streams.get("heartbeat", pd.DataFrame())
if not heartbeat_df.empty and "TimestampSeconds" in heartbeat_df.columns:
    hb_sorted = heartbeat_df.dropna(subset=["TimestampSeconds"]).sort_values("TimestampSeconds")
else:
    hb_sorted = pd.DataFrame()
hb_first, hb_last = _split_first_last(hb_sorted, n_points)
for chunk in (hb_first, hb_last):
    if not chunk.empty:
        chunk["corrected_time"] = chunk["TimestampSeconds"].apply(_correct_seconds)

# Prepare corrected video summary using heartbeat mapping
video_first_raw = video_csv.head(n_points).copy()
video_last_raw = video_csv.tail(n_points).copy()
video_first_corr = video_first_raw.copy()
video_last_corr = video_last_raw.copy()
for chunk in (video_first_corr, video_last_corr):
    chunk["corrected_time"] = chunk["Seconds"].apply(_correct_seconds)

# Raw heartbeat summaries
raw_hb_sorted = pd.DataFrame()
if not raw_heartbeat.empty and "TimestampSeconds" in raw_heartbeat.columns:
    raw_hb_sorted = raw_heartbeat.dropna(subset=["TimestampSeconds"]).sort_values("TimestampSeconds")
raw_hb_first, raw_hb_last = _split_first_last(raw_hb_sorted, n_points)

# Gather register edges
corrected_output_set = streams.get("output_set")
corrected_output_clear = streams.get("output_clear")
corrected_set_edge = _register_edge(corrected_output_set, "first", fallback=raw_output_set)
corrected_clear_edge = _register_edge(corrected_output_clear, "last", fallback=raw_output_clear)
raw_set_edge = _register_edge(raw_output_set, "first")
raw_clear_edge = _register_edge(raw_output_clear, "last")

print(f"Session root: {exp_root}")
print(f"Video CSV used: {video_csv_path.name}")

print("\n=== Corrected data via load_all_streams ===")
_print_table(
    f"Heartbeat first {min(n_points, len(hb_first))} rows",
    hb_first,
    _available_columns(hb_first, ["TimestampSeconds", "corrected_time", "Time"])
 )
_print_table(
    f"Heartbeat last {min(n_points, len(hb_last))} rows",
    hb_last,
    _available_columns(hb_last, ["TimestampSeconds", "corrected_time", "Time"])
 )
video_cols_corr = ["Seconds", "corrected_time"]
if video_timestamp_col:
    video_cols_corr.append(video_timestamp_col)
_print_table(
    f"Video first {min(n_points, len(video_first_corr))} rows (corrected)",
    video_first_corr,
    _available_columns(video_first_corr, video_cols_corr)
 )
_print_table(
    f"Video last {min(n_points, len(video_last_corr))} rows (corrected)",
    video_last_corr,
    _available_columns(video_last_corr, video_cols_corr)
 )
_print_edge("output_set first DOPort0 (corrected)", corrected_set_edge)
_print_edge("output_clear last DOPort0 (corrected)", corrected_clear_edge)

print("\n=== Raw file excerpts (no correction applied) ===")
_print_table(
    f"Heartbeat first {min(n_points, len(raw_hb_first))} rows",
    raw_hb_first,
    _available_columns(raw_hb_first, ["TimestampSeconds", "Time"])
 )
_print_table(
    f"Heartbeat last {min(n_points, len(raw_hb_last))} rows",
    raw_hb_last,
    _available_columns(raw_hb_last, ["TimestampSeconds", "Time"])
 )
video_cols_raw = ["Seconds"]
if video_timestamp_col:
    video_cols_raw.append(video_timestamp_col)
_print_table(
    f"Video first {min(n_points, len(video_first_raw))} rows (raw)",
    video_first_raw,
    _available_columns(video_first_raw, video_cols_raw)
 )
_print_table(
    f"Video last {min(n_points, len(video_last_raw))} rows (raw)",
    video_last_raw,
    _available_columns(video_last_raw, video_cols_raw)
 )
_print_edge("output_set first DOPort0 (raw)", raw_set_edge)
_print_edge("output_clear last DOPort0 (raw)", raw_clear_edge)

In [ ]:
# Quantify load_all_streams timing error and behavior ↔ video alignment
import pandas as pd
import numpy as np
import harp
import hypnose.trial_classification.classification_utils as cu
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# Keep this configuration in sync with the previous cell
subjid = 99
date = 20260303
experiment_index = 4
video_csv_index = 0
alignment_tolerance_ms = 17
# ------------------------------------------------------

exp_root = load_experiment(subjid, date, index=experiment_index)
streams = load_all_streams(exp_root, apply_corrections=True, verbose=False)
real_offset = streams.get("real_time_offset", pd.Timedelta(0))
timestamp_map = streams.get("timestamp_to_time")
video_df = streams.get("video_data")
if not isinstance(timestamp_map, pd.Series) or timestamp_map.empty:
    raise ValueError("timestamp_to_time mapping missing; cannot validate corrections")
if video_df is None or video_df.empty:
    raise ValueError("video_data missing from streams; run load_all_streams first")

def _seconds_to_corrected(seconds_value):
    if pd.isna(seconds_value):
        return pd.NaT
    ts_series = timestamp_map.dropna().sort_index()
    if ts_series.empty:
        return pd.NaT
    sec_idx = ts_series.index.to_numpy(dtype=float)
    time_ns = ts_series.values.astype("datetime64[ns]").astype(np.int64)
    sec_val = float(seconds_value)
    if len(sec_idx) == 1:
        interp_ns = time_ns[0] + int((sec_val - sec_idx[0]) * 1e9)
    elif sec_val <= sec_idx[0]:
        interp_ns = time_ns[0] + int((sec_val - sec_idx[0]) * 1e9)
    elif sec_val >= sec_idx[-1]:
        interp_ns = time_ns[-1] + int((sec_val - sec_idx[-1]) * 1e9)
    else:
        interp_ns = np.interp(sec_val, sec_idx, time_ns)
    return pd.to_datetime(int(interp_ns)) + real_offset

behavior_reader = harp.create_reader(str(cu.BEHAVIOR_SCHEMA_PATH), epoch=harp.REFERENCE_EPOCH)
raw_heartbeat = cu.load(behavior_reader.TimestampSeconds, exp_root / "Behavior")
if raw_heartbeat.empty:
    raise ValueError("Raw heartbeat missing; cannot compare corrections")
raw_heartbeat = raw_heartbeat.reset_index().rename(columns={"index": "Time"})
raw_heartbeat["Time"] = pd.to_datetime(raw_heartbeat["Time"], errors="coerce")
raw_heartbeat = raw_heartbeat.dropna(subset=["Time", "TimestampSeconds"])
raw_heartbeat["offset_corrected"] = raw_heartbeat["Time"] + real_offset
raw_heartbeat["interp_corrected"] = raw_heartbeat["TimestampSeconds"].apply(_seconds_to_corrected)
raw_heartbeat["delta_ms"] = (raw_heartbeat["interp_corrected"] - raw_heartbeat["offset_corrected"]).dt.total_seconds() * 1000
hb_abs_max = raw_heartbeat["delta_ms"].abs().max()
print(f"Heartbeat interpolation delta max |Δ| = {hb_abs_max:.6f} ms")
print(raw_heartbeat[["TimestampSeconds", "delta_ms"]].head(3))

video_dir = exp_root / "VideoData"
video_csvs = sorted(video_dir.glob("*.csv"))
if not video_csvs:
    raise FileNotFoundError(f"No VideoData CSVs in {video_dir}")
if not (0 <= video_csv_index < len(video_csvs)):
    raise IndexError(f"video_csv_index {video_csv_index} out of range (found {len(video_csvs)})")
video_csv_path = video_csvs[video_csv_index]
video_raw = pd.read_csv(video_csv_path)
if "Seconds" not in video_raw.columns:
    raise ValueError(f"'Seconds' column missing in {video_csv_path.name}")
video_raw = video_raw.reset_index().rename(columns={"index": "_frame"})
video_raw["Seconds"] = pd.to_numeric(video_raw["Seconds"], errors="coerce")
video_raw = video_raw.dropna(subset=["Seconds"])
video_raw["corrected_from_seconds"] = video_raw["Seconds"].apply(_seconds_to_corrected)

video_corr = video_df.reset_index()
index_col_name = video_df.index.name if video_df.index.name in video_corr.columns else "index"
if index_col_name not in video_corr.columns:
    raise KeyError(f"Could not find index column after reset_index; columns={list(video_corr.columns)}")
if "corrected_time_lls" in video_corr.columns and index_col_name != "corrected_time_lls":
    pass
else:
    video_corr = video_corr.rename(columns={index_col_name: "corrected_time_lls"})
if "corrected_time_lls" not in video_corr.columns:
    raise KeyError(f"Failed to create corrected_time_lls column; columns now {list(video_corr.columns)}")

merged_video = video_corr.merge(video_raw[['_frame', 'corrected_from_seconds']], on="_frame", how="inner")
corrected_time_col = "corrected_time_lls"
if corrected_time_col not in merged_video.columns:
    candidates = [col for col in merged_video.columns if col.startswith("corrected_time") or col.endswith("corrected_time_lls")]
    if candidates:
        corrected_time_col = candidates[0]
    else:
        raise KeyError(f"Could not locate corrected time column after merge; merged columns: {list(merged_video.columns)}")
merged_video["delta_ms"] = (merged_video[corrected_time_col] - merged_video["corrected_from_seconds"]).dt.total_seconds() * 1000
print(f"Video timestamp delta max |Δ| = {merged_video['delta_ms'].abs().max():.6f} ms")
print(merged_video[["_frame", "delta_ms"]].head(3))

dip0 = streams.get("digital_input_data", {}).get("DIPort0")
if dip0 is None or dip0.empty:
    print("No DIPort0 data available for alignment check")
else:
    tol = pd.Timedelta(milliseconds=alignment_tolerance_ms)
    dip0_bool = dip0.astype(bool).sort_index()
    state_changes = dip0_bool.astype(int).diff().fillna(0)
    behavior_events = state_changes[state_changes != 0].index.to_series().sort_values().reset_index(drop=True)
    video_times = merged_video[corrected_time_col].sort_values().reset_index(drop=True)
    beh_df = pd.DataFrame({"behavior_time": behavior_events})
    vid_df = pd.DataFrame({"video_time": video_times})
    matches = pd.merge_asof(beh_df, vid_df, left_on="behavior_time", right_on="video_time", direction="nearest", tolerance=tol)
    matched = matches["video_time"].notna()
    match_ratio = matched.mean() if not matches.empty else np.nan
    matches.loc[matched, "delta_ms"] = (matches.loc[matched, "behavior_time"] - matches.loc[matched, "video_time"]).dt.total_seconds() * 1000
    print(f"Behavior↔video alignment: {matched.sum()} / {len(matches)} events matched within ±{alignment_tolerance_ms} ms (ratio {match_ratio:.3f})")
    print(matches[matched].head(5))

In [ ]:
# Check clock alignment between Port0 events and video frame timestamps (raw data only). As we record at 60 fps, a perfect match is within ~16.7 ms (thus, tolerance at 17 ms should capture all Port0 matches).
import pandas as pd
from hypnose.trial_classification.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 40
date = 20251128
experiment_index = 0          # pick run if multiple
apply_corrections = True      # use heartbeat/folder offset; set False to stay in device time
tolerance_ms = 17              # max allowed delta for a "match"
# ----------------------

exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; check inputs")
streams = load_all_streams(exp_root, apply_corrections=apply_corrections, verbose=False)

# Port0 timestamps
dip0 = streams.get("digital_input_data", {}).get("DIPort0")
if dip0 is None or dip0.empty:
    raise ValueError("DIPort0 missing or empty")
port0_times = pd.DataFrame({"time_port0": pd.to_datetime(dip0.index)}).dropna().sort_values("time_port0")

# Video frame timestamps
video_df = streams.get("video_data")
if video_df is None or video_df.empty:
    raise ValueError("video_data missing or empty")
video_times = pd.DataFrame({"time_video": pd.to_datetime(video_df.index)}).dropna().sort_values("time_video")

# Merge-asof both directions to check nearest matches within tolerance
tol = pd.Timedelta(milliseconds=tolerance_ms)

v_to_p = pd.merge_asof(video_times, port0_times, left_on="time_video", right_on="time_port0", direction="nearest", tolerance=tol)
p_to_v = pd.merge_asof(port0_times, video_times, left_on="time_port0", right_on="time_video", direction="nearest", tolerance=tol)

unmatched_video = v_to_p["time_port0"].isna().sum()
unmatched_port0 = p_to_v["time_video"].isna().sum()
matched_video = len(v_to_p) - unmatched_video
matched_port0 = len(p_to_v) - unmatched_port0

print(f"Streams loaded from: {exp_root}")
print(f"apply_corrections={apply_corrections}, tolerance={tolerance_ms} ms")
print(f"Video frames: {len(video_times):,}, Port0 samples: {len(port0_times):,}")
print(f"Video frames with match: {matched_video:,} (unmatched: {unmatched_video:,})")
print(f"Port0 samples with match: {matched_port0:,} (unmatched: {unmatched_port0:,})")

# Show first 10 matched pairs (video -> port0) with deltas
matched_rows = v_to_p.dropna(subset=["time_port0"]).head(10).copy()
if not matched_rows.empty:
    matched_rows["delta_ms"] = (matched_rows["time_port0"] - matched_rows["time_video"]).dt.total_seconds() * 1000
    print("\nFirst 10 video->Port0 matches:")
    for _, row in matched_rows.iterrows():
        print(f"video {row['time_video']}  |  port0 {row['time_port0']}  |  delta_ms={row['delta_ms']:.3f}")
else:
    print("No video frames matched within tolerance")

matched_rows_p = p_to_v.dropna(subset=["time_video"]).head(10).copy()
if not matched_rows_p.empty:
    matched_rows_p["delta_ms"] = (matched_rows_p["time_video"] - matched_rows_p["time_port0"]).dt.total_seconds() * 1000
    print("\nFirst 10 Port0->video matches:")
    for _, row in matched_rows_p.iterrows():
        print(f"port0 {row['time_port0']}  |  video {row['time_video']}  |  delta_ms={row['delta_ms']:.3f}")
else:
    print("No Port0 samples matched within tolerance")


In [ ]:
# Inspect SLEAP predictions from .slp for a frame range
import h5py
import numpy as np
import pandas as pd
import json
from pathlib import Path

# --- Configure here ---
slp_path = r"Z:/hypnose/derivatives/sub-040_id-259/ses-040_date-20251218/saved_analysis_results/2025-12-18T15-59-34__VideoData_1904-01-23T04-00-00.predictions.slp"
frame_start = 26057
frame_end = 26080
use_full_range = False   # set True to ignore frame_start/frame_end and return all frames
node_idx_max = 4        # only include node indices 0..node_idx_max
instance_idx = 0        # dense layout only
# ----------------------

slp_path = Path(slp_path)
if not slp_path.exists():
    raise FileNotFoundError(slp_path)

debug_path = slp_path.parent / debug_csv_name
with h5py.File(slp_path, "r") as f:
    pts = f["pred_points"]
    conf = f.get("pred_confidence")
    frames_ds = f.get("frames")
    node_names = None
    if "node_names" in f:
        try:
            node_names = [n.decode("utf-8") if isinstance(n, (bytes, np.bytes_)) else str(n) for n in f["node_names"][:]]
        except Exception:
            node_names = None

    ndim = pts.ndim
    shape = pts.shape
    dtype_names = list(pts.dtype.names) if pts.dtype.names else []
    print(f"pred_points shape: {shape}")
    if dtype_names:
        print(f"pred_points dtype fields: {dtype_names}")
    if conf is not None:
        print(f"pred_confidence shape: {conf.shape}")

    # Build frame/offset mapping (mimic sleap_utils logic)
    frames_df = None
    video_lookup = {}
    offsets = {}
    if frames_ds is not None and frames_ds.dtype.names:
        frames_fields = list(frames_ds.dtype.names or [])
        frame_num_field = next((n for n in frames_fields if n in ("frame_idx", "frame_number", "frame")), None)
        video_field = next((n for n in frames_fields if n in ("video", "video_id", "video_idx")), None)
        frames_arr = frames_ds[:]
        frames_df = pd.DataFrame({"frame_id": np.arange(len(frames_arr))})
        frames_df["frame_local"] = frames_arr[frame_num_field].astype(int) if frame_num_field else frames_df["frame_id"]
        frames_df["video_idx"] = frames_arr[video_field].astype(int) if video_field else 0
        # Video names from videos_json if present
        vids = None
        if "videos_json" in f:
            try:
                vids_raw = f["videos_json"][()]
                vids = json.loads(vids_raw.decode("utf-8") if isinstance(vids_raw, (bytes, np.bytes_)) else vids_raw)
            except Exception:
                vids = None
        if isinstance(vids, list):
            for i, v in enumerate(vids):
                name = None
                if isinstance(v, dict):
                    name = v.get("filename") or v.get("file")
                if not name and isinstance(v, str):
                    name = v
                if name:
                    video_lookup[i] = Path(name).name
        # Offsets per video (ordered by video_idx) for global frame numbering
        total = 0
        for vid in sorted(frames_df["video_idx"].unique()):
            offsets[vid] = total
            sub = frames_df[frames_df["video_idx"] == vid]
            if not sub.empty and sub["frame_local"].notna().any():
                total += int(sub["frame_local"].max()) + 1
            else:
                total += len(sub)
        print(f"frames dataset fields: {frames_fields}")
        print(f"video offsets: {offsets}")

    df_slice = pd.DataFrame()
    structured_handled = False
    # Structured flat layout: pred_points is 1-D with fields like x/y/score and instances reference slices.
    if ndim == 1 and dtype_names and {"x", "y"}.issubset(dtype_names):
        inst_ds = f.get("instances")
        if inst_ds is None:
            raise ValueError("Structured pred_points but instances dataset missing; cannot map to frames")
        inst_fields = list(inst_ds.dtype.names or [])
        frame_field = next((n for n in inst_fields if n in ("frame_id", "frame", "frame_idx")), None)
        start_field = next((n for n in inst_fields if "point_id_start" in n or ("point" in n and "start" in n)), None)
        end_field = next((n for n in inst_fields if "point_id_end" in n or ("point" in n and "end" in n)), None)
        if not all([frame_field, start_field, end_field]):
            print(f"instances dtype fields: {inst_fields}")
            raise ValueError("Could not find frame/start/end fields in instances dataset")

        inst_arr = inst_ds[:]
        inst_df = pd.DataFrame({
            "frame_id": inst_arr[frame_field].astype(int),
            "start": inst_arr[start_field].astype(int),
            "end": inst_arr[end_field].astype(int),
            "instance_id": inst_arr["instance_id"].astype(int) if "instance_id" in inst_fields else np.arange(len(inst_arr)),
        })
        inst_df["count"] = inst_df["end"] - inst_df["start"]
        if frames_df is not None:
            inst_df = inst_df.join(frames_df.set_index("frame_id"), on="frame_id", how="left")
            inst_df["video_file"] = inst_df["video_idx"].map(video_lookup) if video_lookup else None
            inst_df["global_frame"] = inst_df["frame_local"] + inst_df["video_idx"].map(offsets)
        # Filter range using global_frame if present, else frame_local, else frame_id
        if use_full_range:
            mask = inst_df["count"] > 0
        elif "global_frame" in inst_df.columns and inst_df["global_frame"].notna().any():
            mask = (inst_df["global_frame"] >= frame_start) & (inst_df["global_frame"] <= frame_end) & (inst_df["count"] > 0)
        elif "frame_local" in inst_df.columns and inst_df["frame_local"].notna().any():
            mask = (inst_df["frame_local"] >= frame_start) & (inst_df["frame_local"] <= frame_end) & (inst_df["count"] > 0)
        else:
            mask = (inst_df["frame_id"] >= frame_start) & (inst_df["frame_id"] <= frame_end) & (inst_df["count"] > 0)
        inst_df = inst_df[mask]
        if inst_df.empty:
            print("No instances in requested frame range")
        else:
            rows = []
            for _, inst in inst_df.reset_index(drop=True).iterrows():
                start = int(inst["start"])
                end = int(inst["end"])
                pt_slice = pts[start:end]
                for node_idx, pt in enumerate(pt_slice):
                    if node_idx_max is not None and node_idx > node_idx_max:
                        break
                    rows.append({
                        "frame": int(inst.get("global_frame", inst.get("frame_local", inst.get("frame_id", 0)))),
                        "node_idx": int(node_idx),
                        "x": float(pt["x"]),
                        "y": float(pt["y"]),
                        "score": float(pt["score"]) if "score" in dtype_names else np.nan,
                    })
            df_slice = pd.DataFrame(rows, columns=["frame", "node_idx", "x", "y", "score"])
            print(f"Structured layout rows: {len(df_slice)}")
            structured_handled = True

    if structured_handled:
        pass
    else:
        # Guard against unexpected storage layouts
        if ndim not in (3, 4):
            print("Unexpected pred_points ndim; listing file keys for inspection:")
            print(list(f.keys()))
            try:
                arr_sample = pts[: min(5, pts.shape[0])]
                print(f"Sample dtype: {arr_sample.dtype}, sample shape: {arr_sample.shape}")
            except Exception as e:
                print(f"Could not sample pred_points: {e}")
            raise ValueError(f"Unexpected pred_points ndim={ndim}")

        # Slice frames (dense layout)
        fs = 0 if use_full_range else max(frame_start, 0)
        fe = shape[0] - 1 if use_full_range else min(frame_end, shape[0] - 1)
        if fe < fs:
            raise ValueError("frame_end before frame_start or out of range")

        if ndim == 4:  # (frames, instances, nodes, 2)
            pts_slice = pts[fs:fe+1, instance_idx, :, :]
            conf_slice = conf[fs:fe+1, instance_idx, :] if conf is not None else None
            nodes = pts_slice.shape[1]
        elif ndim == 3:  # (frames, nodes, 2)
            pts_slice = pts[fs:fe+1, :, :]
            conf_slice = conf[fs:fe+1, :] if conf is not None else None
            nodes = pts_slice.shape[1]
        max_nodes = min(nodes, node_idx_max + 1) if node_idx_max is not None else nodes

        # Build rows (long form)
        rows = []
        frame_vals = np.arange(fs, fe + 1)
        for fi, frame in enumerate(frame_vals):
            for ni in range(max_nodes):
                score_val = float(conf_slice[fi, ni]) if conf_slice is not None else np.nan
                rows.append({
                    "frame": int(frame),
                    "node_idx": int(ni),
                    "x": float(pts_slice[fi, ni, 0]),
                    "y": float(pts_slice[fi, ni, 1]),
                    "score": score_val,
                })
        df_slice = pd.DataFrame(rows, columns=["frame", "node_idx", "x", "y", "score"])
        print(f"Dense layout frames {fs}-{fe}, instance {instance_idx}, nodes {max_nodes}")

    if not df_slice.empty:
        df_slice.to_csv(debug_path, index=False)
        print(f"Saved debug CSV: {debug_path}")
        print(df_slice.head(20))
    else:
        print("No data to save")